## ⚠ Save a copy to your Drive first

**This notebook is fetched fresh from GitHub every time you open the link.** Any edits you make here — settings, code, hyperparameters — **will be LOST when you close the tab** unless you save a copy.

**To keep your edits:**

1. **File → Save a copy in Drive** (top menu)
2. Re-open the saved copy via **File → Open notebook → Recent** or your Google Drive next time

The saved copy is yours to edit; the GitHub link always opens fresh.

# Train a Tabular Classifier for IGNODE

**Maintained by:** IGNODE  
**Last verified:** 2026-05-30 against Python 3.11, LightGBM 4.5, scikit-learn 1.5  
**Runtime:** under 2 minutes on Colab's free CPU tier (no GPU needed)

Train a tabular classification model — predict categories like `Yes/No`, `Setosa/Versicolor/Virginica`, or `Normal/Warning/Critical`. Choose between a single algorithm or AutoML in the **Settings** cell.

## What you'll need

- A `.csv` file with a **header row**, one column per feature, and one **label column**
- A Google account to run this notebook in Colab

## What this notebook produces

1. `model.onnx` — the trained model in ONNX format
2. `class_labels.json` + `feature_columns.json` — sidecar metadata
3. The exact strings you'll paste into the IGNODE Custom Model Upload wizard

## Steps

1. **Runtime → Run all** (top menu) — or run each cell in order with Shift+Enter
2. When the upload cell prompts, drop your CSV file
3. Edit the **Settings** cell if you want a different algorithm or hyperparameters
4. After training, download `model.onnx` and the metadata, then go to **IGNODE → ML Factory → Custom Models → + Upload ML Model**

## Quick start

### To try it right now (no setup needed)

1. **Runtime → Run all** at the top of Colab
2. Wait ~30 seconds for dependencies + ~60 seconds for AutoML to explore 6 algorithms
3. The last cell automatically downloads `model.onnx` + sidecar JSONs to your laptop — that's the AutoML winner on the sample IoT data

### To train on YOUR data

You only need to edit **two values**:

| Step | Cell | What to change |
|---|---|---|
| 1 | **Load data** cell | `SAMPLE_DATASET = 'sensor_anomaly_classification'` → `SAMPLE_DATASET = None` |
| 2 | **Settings** cell | `LABEL_COLUMN = 'Anomaly'` → `LABEL_COLUMN = 'your_target_column'` |

### Three ways to control which algorithm trains

This notebook accepts three shapes for `ALGORITHM` in the Settings cell — see that cell for the full table. Quick version:

| `ALGORITHM` value | Result |
|---|---|
| `"auto"` | AutoML over 6 algorithms (default — best quality) |
| `"lightgbm"` (or another name) | Train just that algorithm — fast |
| `["lightgbm", "xgboost"]` | Custom sweep — train each, pick the winner |

### What you get at the end

- `model.onnx` — the winning model
- `class_labels.json` — class names
- `feature_columns.json` — input contract (`{feature_columns, label_columns}`)

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

In [ ]:
!pip install --quiet \
    lightgbm==4.5.0 \
    xgboost==2.1.2 \
    scikit-learn==1.5.2 \
    flaml==2.3.2 \
    onnx==1.21.0 \
    onnxmltools==1.16.0 \
    skl2onnx==1.20.0 \
    onnxconverter-common==1.14.0 \
    onnxruntime==1.23.2

import sys
import lightgbm as lgb
import xgboost as xgb
import sklearn
import flaml
import onnx
print(f'Python:       {sys.version.split()[0]}')
print(f'LightGBM:     {lgb.__version__}')
print(f'XGBoost:      {xgb.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'FLAML:        {flaml.__version__}')
print(f'onnx:         {onnx.__version__}')

In [ ]:
!pip install --quiet \
    lightgbm==4.5.0 \
    xgboost==2.1.2 \
    scikit-learn==1.5.2 \
    flaml==2.3.2 \
    onnx==1.21.0 \
    onnxmltools==1.16.0 \
    skl2onnx==1.20.0 \
    onnxruntime==1.23.2

import sys
import lightgbm as lgb
import xgboost as xgb
import sklearn
import flaml
import onnx
print(f'Python:       {sys.version.split()[0]}')
print(f'LightGBM:     {lgb.__version__}')
print(f'XGBoost:      {xgb.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'FLAML:        {flaml.__version__}')
print(f'onnx:         {onnx.__version__}')

## 2. Load data

This cell ships set to a small **IoT sample dataset** so the notebook runs end-to-end out of the box — just hit **Runtime → Run all** and you'll have a trained model in a couple of minutes.

When you're ready to use your own data:

1. Set `SAMPLE_DATASET = None` in the cell below — the cell will then prompt you to upload a CSV
2. Update `LABEL_COLUMN` in the Settings cell further down to the column you want to predict

The default sample is `sensor_anomaly_classification.csv` — ~90 rows of sensor readings (Temperature / Humidity / Vibration / Pressure / RPM / PowerConsumption / SoundLevel / OperatingHours) with an `Anomaly` label (Normal / Warning / Critical).

In [ ]:
# ───────── EDIT THIS ─────────
# The notebook ships with a sample IoT dataset so it runs end-to-end.
# Set SAMPLE_DATASET = None to upload your own CSV instead.
SAMPLE_DATASET = 'sensor_anomaly_classification'
# Available samples (classification):
#   'sensor_anomaly_classification'  — sensor readings, label='Anomaly'
# ────────────────────────────

import pandas as pd

if SAMPLE_DATASET:
    url = f'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/{SAMPLE_DATASET}.csv'
    df = pd.read_csv(url)
    print(f'Loaded sample {SAMPLE_DATASET!r}: {df.shape[0]} rows x {df.shape[1]} columns')
else:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))
    df = pd.read_csv(csv_path)
    print(f'Loaded {csv_path}: {df.shape[0]} rows x {df.shape[1]} columns')

## 3. Inspect the data

Quick look at the columns, value distribution, and dtypes. If anything looks wrong (wrong column names, label column with all same value, missing data), fix the CSV and re-upload before training.

In [ ]:
print('First 5 rows:')
display(df.head())

print('\nColumn dtypes:')
print(df.dtypes)

print('\nStatistics for numeric columns:')
display(df.describe())

## 4. Settings — pick your algorithm(s)

**Edit this cell to match your dataset.** Most users only touch `LABEL_COLUMN` and (optionally) `ALGORITHM`.

### Algorithm list (aligned with IGNODE's in-platform AutoML)

These are the sklearn equivalents of the algorithms IGNODE's in-platform AutoML tries. A model trained here should perform comparably to one trained in ML Factory's ML Jobs tab.

| Name | sklearn / library equivalent | Mirrors IGNODE in-platform AutoML algorithm |
|---|---|---|
| `lightgbm` | LightGBM gradient-boosted trees | `LightGBM` |
| `xgboost` | XGBoost gradient-boosted trees | `FastTree` |
| `random_forest` | RandomForest | `FastForest` |
| `extra_trees` | ExtraTrees | _(no direct equivalent — adds diversity)_ |
| `logistic_regression` | LogisticRegression with L2 penalty | `LogisticRegression` |
| `logistic_regression_l1` | LogisticRegression with L1 (sparse) penalty | `SDCA` |

### `ALGORITHM` accepts three shapes

| Shape | Value | What runs |
|---|---|---|
| **AutoML** | `"auto"` | FLAML hyperparameter search over **all six** algorithms above — same mental model as IGNODE's in-platform AutoML leaderboard |
| **Single** | `"lightgbm"` (or any one name from the list) | Just that algorithm with sensible defaults — fast direct fit, no AutoML overhead |
| **Custom sweep** | `["lightgbm", "xgboost"]` (any list of 2+ names) | Train each with defaults, compare on held-out test set, pick the winner |

### Hyperparameters

- `AUTOML_TIME_BUDGET_SECONDS` — how long AutoML explores (only used when `ALGORITHM="auto"`). 30–120 seconds is plenty for typical CSVs.

In [ ]:
# ───────── EDIT THESE ─────────
# Default = the sample dataset's label column ('Anomaly' for sensor_anomaly_classification).
# When you switch SAMPLE_DATASET = None above and upload your own CSV, change this
# to whichever column you want to predict.
LABEL_COLUMN = 'Anomaly'

# 'auto' = AutoML over all six algorithms
# Single string = direct fit (e.g. 'lightgbm', 'xgboost', 'random_forest', 'extra_trees',
#                              'logistic_regression', 'logistic_regression_l1')
# List = custom sweep of 2+ algorithms (e.g. ['lightgbm', 'xgboost'])
ALGORITHM = 'auto'

TEST_SIZE = 0.2
RANDOM_SEED = 42
AUTOML_TIME_BUDGET_SECONDS = 60   # only used when ALGORITHM='auto'
# ──────────────────────────────

# Sanity-check the label column.
if LABEL_COLUMN not in df.columns:
    raise ValueError(
        f"Label column '{LABEL_COLUMN}' not found in CSV. "
        f"Available columns: {list(df.columns)}"
    )

# Normalize ALGORITHM into one of three modes for the training cell to dispatch on:
#   mode 'auto'   → FLAML AutoML
#   mode 'single' → direct fit of one algorithm
#   mode 'sweep'  → manual sweep over a list, pick the winner on test set
SUPPORTED_ALGOS = {
    'lightgbm', 'xgboost', 'random_forest', 'extra_trees',
    'logistic_regression', 'logistic_regression_l1',
}

if ALGORITHM == 'auto':
    algo_mode = 'auto'
    algo_value = None
elif isinstance(ALGORITHM, str):
    if ALGORITHM not in SUPPORTED_ALGOS:
        raise ValueError(
            f"ALGORITHM={ALGORITHM!r} is not recognized. "
            f"Use 'auto', one of {sorted(SUPPORTED_ALGOS)}, or a list of 2+ of those."
        )
    algo_mode = 'single'
    algo_value = ALGORITHM
elif isinstance(ALGORITHM, (list, tuple)):
    if len(ALGORITHM) < 2:
        raise ValueError("List form of ALGORITHM needs at least 2 entries; use a string for a single algorithm.")
    bad = [a for a in ALGORITHM if a not in SUPPORTED_ALGOS]
    if bad:
        raise ValueError(f"Unrecognized algorithm(s) {bad}. Valid: {sorted(SUPPORTED_ALGOS)}")
    if len(set(ALGORITHM)) != len(ALGORITHM):
        raise ValueError(f"Duplicate algorithm in list: {ALGORITHM}")
    algo_mode = 'sweep'
    algo_value = list(ALGORITHM)
else:
    raise TypeError(
        f"ALGORITHM must be 'auto', a string, or a list — got {type(ALGORITHM).__name__}"
    )

print(f"Label column: '{LABEL_COLUMN}'")
print(f"Mode:         {algo_mode}{' (' + str(algo_value) + ')' if algo_value else ''}")
print('\nClass distribution:')
print(df[LABEL_COLUMN].value_counts())

## 5. Prep the data

Auto-rename column names with spaces / punctuation (IGNODE's upload wizard rejects those), encode string class labels as integers, and split into train + test sets.

In [ ]:
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def normalize(name):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')

rename_map = {c: normalize(c) for c in df.columns if c != normalize(c)}
if rename_map:
    print('Renamed columns (apply same fix in your source CSV for clarity):')
    for old, new in rename_map.items():
        print(f'  {old!r}  ->  {new!r}')
    df = df.rename(columns=rename_map)
    if LABEL_COLUMN in rename_map:
        LABEL_COLUMN = rename_map[LABEL_COLUMN]

X = df.drop(columns=[LABEL_COLUMN])
y_raw = df[LABEL_COLUMN]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_labels = [str(c) for c in label_encoder.classes_]
feature_columns = list(X.columns)

print(f'Features ({len(feature_columns)}): {feature_columns}')
print(f'Classes  ({len(class_labels)}):  {class_labels}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape[0]} rows. Test: {X_test.shape[0]} rows.')

## 6. Train

Branches on the `ALGORITHM` you picked in the Settings cell:

- **`auto`** runs FLAML over a whitelist of algorithms that all have reliable ONNX exporters. You'll see a per-trial log and a final winner.
- **Single-algorithm options** train one model with sensible defaults.

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Maps the user-facing algorithm name to:
#   - FLAML's internal name (for 'auto' mode)
#   - a factory that builds a sklearn-compatible classifier with sensible defaults
#     (for 'single' + 'sweep' modes)
def _build(name):
    if name == 'lightgbm':
        return 'lgbm', lgb.LGBMClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=-1,
            random_state=RANDOM_SEED, verbose=-1,
        )
    if name == 'xgboost':
        return 'xgboost', xgb.XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=6,
            random_state=RANDOM_SEED, eval_metric='mlogloss',
        )
    if name == 'random_forest':
        return 'rf', RandomForestClassifier(
            n_estimators=200, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1,
        )
    if name == 'extra_trees':
        return 'extra_tree', ExtraTreesClassifier(
            n_estimators=200, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1,
        )
    if name == 'logistic_regression':
        return 'lrl2', LogisticRegression(
            penalty='l2', max_iter=1000, random_state=RANDOM_SEED,
        )
    if name == 'logistic_regression_l1':
        return 'lrl1', LogisticRegression(
            penalty='l1', solver='liblinear', max_iter=1000, random_state=RANDOM_SEED,
        )
    raise ValueError(f"Unknown algorithm: {name}")


t0 = time.time()
winner_name = None   # 'lgbm' | 'xgboost' | 'rf' | 'extra_tree' | 'lrl1' | 'lrl2'
winner_model = None  # the trained estimator the ONNX export cell will use

if algo_mode == 'auto':
    # AutoML over all six algorithms. No stacking / blending — winner is always
    # a single, ONNX-exportable model.
    from flaml import AutoML
    automl = AutoML()
    automl.fit(
        X_train=X_train,
        y_train=y_train,
        task='classification',
        time_budget=AUTOML_TIME_BUDGET_SECONDS,
        estimator_list=['lgbm', 'xgboost', 'rf', 'extra_tree', 'lrl1', 'lrl2'],
        metric='accuracy',
        seed=RANDOM_SEED,
        verbose=1,
    )
    winner_name = automl.best_estimator
    winner_model = automl.model.estimator
    print(f'\nAutoML winner: {winner_name}')
    print(f'AutoML best validation accuracy: {1 - automl.best_loss:.3f}')

elif algo_mode == 'single':
    winner_name, winner_model = _build(algo_value)
    winner_model.fit(X_train, y_train)
    print(f'Trained {algo_value} ({winner_name}).')

elif algo_mode == 'sweep':
    # Train each requested algorithm with defaults; pick the one with the highest
    # test-set accuracy as the winner. This is the in-platform-AutoML leaderboard
    # mental model, scoped to the algorithms the customer named.
    print('Sweep results:')
    print(f"  {'Algorithm':<25} {'test accuracy':<15} {'fit time (sec)'}")
    print('  ' + '─' * 60)
    best_score = -float('inf')
    for name in algo_value:
        flaml_name, model = _build(name)
        t_fit = time.time()
        model.fit(X_train, y_train)
        fit_sec = time.time() - t_fit
        score = accuracy_score(y_test, model.predict(X_test))
        marker = ' ←' if score > best_score else ''
        print(f"  {name:<25} {score:<15.3f} {fit_sec:.1f}{marker}")
        if score > best_score:
            best_score = score
            winner_name = flaml_name
            winner_model = model
    print(f'\nSweep winner: {winner_name} (test accuracy {best_score:.3f})')

print(f'\nTotal elapsed: {time.time() - t0:.1f} sec')

## 7. Evaluate

Accuracy, per-class precision / recall / F1, and a confusion matrix.

**Read the confusion matrix:** rows = actual class, columns = predicted class. The diagonal is correct predictions. Off-diagonal bright spots show which classes the model confuses.

In [ ]:
from onnxconverter_common.data_types import FloatTensorType
import onnx
import os

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]

ARTIFACT_FILE = None   # set below — 'model.onnx', 'model.txt', or 'model.json'


def _export(converter, model_obj, opsets=(18, 15, 12)):
    """Try ONNX export at each opset; return the proto on first success,
    or None if all attempts fail (so the caller can try a native fallback)."""
    last_err = None
    for opset in opsets:
        try:
            m = converter(model_obj, initial_types=initial_types, target_opset=opset)
            print(f'  ONNX exported at opset {opset}')
            return m
        except Exception as ex:
            last_err = ex
            print(f'  ONNX opset {opset} failed ({type(ex).__name__}); trying lower')
    return None


# Dispatch by the FLAML-side winner name so the auto / single / sweep modes
# all land on the right ONNX converter.
onnx_model = None
if winner_name == 'lgbm':
    from onnxmltools.convert import convert_lightgbm
    onnx_model = _export(convert_lightgbm, winner_model)
elif winner_name == 'xgboost':
    from onnxmltools.convert import convert_xgboost
    onnx_model = _export(convert_xgboost, winner_model)
elif winner_name in ('rf', 'extra_tree', 'lrl1', 'lrl2'):
    from skl2onnx import convert_sklearn
    onnx_model = _export(convert_sklearn, winner_model)
else:
    raise RuntimeError(f'Unsupported winner type for ONNX export: {winner_name}')

if onnx_model is not None:
    onnx.save_model(onnx_model, 'model.onnx')
    ARTIFACT_FILE = 'model.onnx'
    print(f'\nSaved model.onnx ({len(onnx_model.SerializeToString()) / 1024:.1f} KB)')
else:
    # ─── Native fallback (LightGBM + XGBoost only) ───
    # IGNODE's Custom Model Upload wizard accepts these formats first-class
    # (lgbm_text + xgb_json loaders). Sklearn winners have no equivalent
    # fallback — if that happens, retry with ALGORITHM = "lightgbm" or
    # "xgboost" to get into a fallback-capable algorithm.
    if winner_name == 'lgbm':
        winner_model.booster_.save_model('model.txt')
        ARTIFACT_FILE = 'model.txt'
        print(f'\n⚠ All ONNX opsets failed; saved LightGBM native model.txt ({os.path.getsize("model.txt") / 1024:.1f} KB)')
        print('  When uploading, the wizard will detect the LightGBM text format automatically.')
    elif winner_name == 'xgboost':
        winner_model.save_model('model.json')
        ARTIFACT_FILE = 'model.json'
        print(f'\n⚠ All ONNX opsets failed; saved XGBoost native model.json ({os.path.getsize("model.json") / 1024:.1f} KB)')
        print('  When uploading, the wizard will detect the XGBoost JSON format automatically.')
    else:
        raise RuntimeError(
            f'ONNX export failed for winner {winner_name!r} and no native fallback is available '
            f'for sklearn-style algorithms. Retry with ALGORITHM = "lightgbm" or "xgboost" to get '
            f'a fallback-capable algorithm, or try the simple/ notebook variants.'
        )

## 8. Export to ONNX

Each algorithm has its own ONNX converter. The cell below picks the right one based on the winner type (so AutoML and single-algorithm paths produce equivalent ONNX files for the same algorithm).

Opset 18 first (matches IGNODE's pinset). Falls back to 15 then 12 if a converter doesn't support 18 yet.

In [ ]:
from onnxconverter_common.data_types import FloatTensorType
import onnx

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]


def _export(converter, model_obj, opsets=(18, 15, 12)):
    last_err = None
    for opset in opsets:
        try:
            m = converter(model_obj, initial_types=initial_types, target_opset=opset)
            print(f'  exported at opset {opset}')
            return m
        except Exception as ex:
            last_err = ex
            print(f'  opset {opset} failed ({type(ex).__name__}); trying lower')
    raise RuntimeError(f'All opset attempts failed. Last: {last_err}')


# Dispatch by the FLAML-side winner name so the auto / single / sweep modes
# all land on the right ONNX converter.
if winner_name == 'lgbm':
    from onnxmltools.convert import convert_lightgbm
    onnx_model = _export(convert_lightgbm, winner_model)
elif winner_name == 'xgboost':
    from onnxmltools.convert import convert_xgboost
    onnx_model = _export(convert_xgboost, winner_model)
elif winner_name in ('rf', 'extra_tree', 'lrl1', 'lrl2'):
    from skl2onnx import convert_sklearn
    onnx_model = _export(convert_sklearn, winner_model)
else:
    raise RuntimeError(f'Unsupported winner type for ONNX export: {winner_name}')

onnx.save_model(onnx_model, 'model.onnx')
size_kb = len(onnx_model.SerializeToString()) / 1024
print(f'\nSaved model.onnx ({size_kb:.1f} KB)')

## 9. Write sidecar files

IGNODE's inference runtime reads two sidecar JSON files alongside the model:

- `class_labels.json` — class names in the order the model emits them
- `feature_columns.json` — `{feature_columns, label_columns}` — the input contract

In [ ]:
from google.colab import files
files.download(ARTIFACT_FILE)   # 'model.onnx' / 'model.txt' (LightGBM) / 'model.json' (XGBoost)
files.download('class_labels.json')
files.download('feature_columns.json')

## 10. Download the model

The cells below trigger downloads to your laptop. If your browser blocks them, the files are also visible in the Colab file panel on the left — right-click → Download.

In [ ]:
from google.colab import files

files.download('model.onnx')
files.download('class_labels.json')
files.download('feature_columns.json')

---

## Reusing this notebook for your own data

Two edits switch to your own dataset:

```python
# In the "Load data" cell:
SAMPLE_DATASET = None        # was 'sensor_anomaly_classification'

# In the "Settings" cell:
LABEL_COLUMN = 'YourColumn'  # was 'Anomaly' — whatever you want to predict
```

Everything else adapts automatically.

### Picking the right ALGORITHM mode

| Your goal | Set `ALGORITHM` to |
|---|---|
| Best possible model, willing to wait ~60 sec | `"auto"` (AutoML over all 6) |
| Fast result with strong default | `"lightgbm"` |
| Compare 2-3 specific algorithms | `["lightgbm", "xgboost", "random_forest"]` |
| Linear baseline only | `"logistic_regression"` |
| Sparse linear baseline | `"logistic_regression_l1"` |

### Other optional tweaks (Settings cell)

| Want to change | Edit |
|---|---|
| AutoML time budget | `AUTOML_TIME_BUDGET_SECONDS = 120` (more time = better odds of finding stronger model) |
| Train/test ratio | `TEST_SIZE = 0.3` |
| Reproducibility | `RANDOM_SEED = <any int>` |

### Bringing this code into your own project

The code is vanilla Python except for two Colab helpers (`files.upload()`, `files.download()`). To run outside Colab:

1. Replace the `files.upload()` block with `df = pd.read_csv('/your/local/path.csv')`
2. Replace the `files.download(...)` calls with whatever your project does with output files
3. Everything else works as-is.

### Common errors

| Error | Fix |
|---|---|
| `Label column 'X' not in CSV` | Check the column list printed by the inspect cell |
| `ALGORITHM=X is not recognized` | Use one of the names in the algorithm table above |
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns shown in the prep cell |
| Upload rejected: "duplicate name" | Pick a different name in the upload wizard, or delete the existing model |
| AutoML picked a weak algorithm | Increase `AUTOML_TIME_BUDGET_SECONDS` to give it more exploration time |

## 11. Upload to IGNODE

Open your IGNODE portal and:

1. **Integrations → ML Factory**
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model** (or open the **Add Model** dropdown → **Upload Custom Model**)
4. Drop your `model.onnx` file
5. In the metadata form:
    - **Task Type:** `Classification`
    - **Class Labels:** paste from `class_labels.json`
    - **Feature Columns:** paste from `feature_columns.json` → `feature_columns`
    - **Target Column:** paste from `feature_columns.json` → `label_columns` (single entry)
6. Review and click **Upload**

Then click **Open in Playground** to test it with a sample row from your test data.

### Troubleshooting

| Error | Fix |
|---|---|
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns from Step 5 |
| Upload rejected: "duplicate name" | A model with that name already exists in your org — pick a different name or delete the existing one |
| Playground says "Field X required" | `LABEL_COLUMN` was set wrong. Re-train with the correct value |
| Predictions look wrong | Try `ALGORITHM="auto"` to let AutoML explore more algorithms, or increase `AUTOML_TIME_BUDGET_SECONDS` |

Need to retrain with different settings? Edit the **Settings** cell and **Runtime → Run all** again.